In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings

warnings.filterwarnings('ignore')
print('Librerias cargadas correctamente.')

Librerias cargadas correctamente.


In [2]:
import os
import pickle

ruta_local = 'models/'

if os.path.exists(ruta_local + 'mejor_modelo.pkl'):
    ruta = ruta_local
    print('Cargando modelo desde el repositorio local (carpeta models/).')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    ruta = '/content/drive/MyDrive/TFM_xG/'
    print('Cargando modelo desde Google Drive.')

with open(ruta + 'mejor_modelo.pkl', 'rb') as f:
    modelo = pickle.load(f)

with open(ruta + 'features.pkl', 'rb') as f:
    features = pickle.load(f)

with open(ruta + 'mejor_modelo_nombre.pkl', 'rb') as f:
    mejor_nombre = pickle.load(f)

print(f'Modelo cargado: {mejor_nombre}')
print(f'Variables esperadas: {len(features)}')
print(features)

Mounted at /content/drive
Cargando modelo desde Google Drive.
Modelo cargado: Regresion Logistica
Variables esperadas: 14
['shot_distance', 'shot_angle', 'under_pressure', 'shot_first_time', 'minute', 'shot_body_part_Left Foot', 'shot_body_part_Other', 'shot_body_part_Right Foot', 'shot_technique_Diving Header', 'shot_technique_Half Volley', 'shot_technique_Lob', 'shot_technique_Normal', 'shot_technique_Overhead Kick', 'shot_technique_Volley']


Obtenemos el modelo ya entrenado y las 14 variables que necesita para funcionar. La revisión confirma que todo se cargó correctamente y que el modelo está listo para hacer predicciones con nuevos datos.

In [3]:
GOAL_X, GOAL_Y = 120, 40
GOAL_POST_Y1, GOAL_POST_Y2 = 36, 44

BODY_PARTS = ['Left Foot', 'Other', 'Right Foot']
TECHNIQUES  = ['Diving Header', 'Half Volley', 'Lob',
               'Normal', 'Overhead Kick', 'Volley']

def calcular_distancia(x, y):
    return np.sqrt((x - GOAL_X)**2 + (y - GOAL_Y)**2)

def calcular_angulo(x, y):
    a = np.array([x - GOAL_X, y - GOAL_POST_Y1])
    b = np.array([x - GOAL_X, y - GOAL_POST_Y2])
    cos_angle = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))

def predecir_xg(x, y,
                body_part='Right Foot',
                technique='Normal',
                under_pressure=False,
                shot_first_time=False,
                minute=45):
    """
    Predice la probabilidad de gol (xG) de un disparo.

    Parametros:
        x, y          : Coordenadas del disparo (sistema StatsBomb: 0-120, 0-80)
        body_part     : Parte del cuerpo ('Right Foot', 'Left Foot', 'Head', 'Other')
        technique     : Tecnica ('Normal', 'Volley', 'Lob', 'Half Volley',
                        'Overhead Kick', 'Diving Header')
        under_pressure: True si el jugador estaba presionado al disparar
        shot_first_time: True si fue un disparo a la primera
        minute        : Minuto del partido

    Devuelve:
        xg (float): Probabilidad estimada de gol entre 0 y 1
    """

    distancia = calcular_distancia(x, y)
    angulo    = calcular_angulo(x, y)


    fila = {
        'shot_distance':   distancia,
        'shot_angle':      angulo,
        'under_pressure':  int(under_pressure),
        'shot_first_time': int(shot_first_time),
        'minute':          minute,
    }


    for bp in BODY_PARTS:
        fila[f'shot_body_part_{bp}'] = int(body_part == bp)


    for tech in TECHNIQUES:
        fila[f'shot_technique_{tech}'] = int(technique == tech)


    X_input = pd.DataFrame([fila])[features]


    xg = modelo.predict_proba(X_input)[0, 1]
    return round(float(xg), 4)

print('Funcion predecir_xg() definida correctamente.')
print('Lista para usar.')

Funcion predecir_xg() definida correctamente.
Lista para usar.


En esta parte se diseña la función principal del proyecto. Se toman en cuenta las coordenadas del disparo, la parte del cuerpo que se usó, la técnica y el contexto del partido. Esta función calcula la distancia y el ángulo hacia la portería. Después se arma un conjunto de datos en el formato que necesita el modelo y obtiene la probabilidad de que el disparo sea gol (xG), que va de 0 a 1. Esta es la pieza clave que transforma el modelo entrenado en una herramienta lista para analizar cualquier disparo nuevo. También nos aseguramos de que toda funciona correctamente.

In [4]:
print('=' * 55)
print('EJEMPLOS DE PREDICCION xG')
print('=' * 55)

casos = [
    {
        'descripcion': 'Remate de cabeza al borde del area',
        'params': dict(x=105, y=40, body_part='Head',
                      technique='Normal', under_pressure=True,
                      shot_first_time=True, minute=60)
    },
    {
        'descripcion': 'Disparo con pie derecho desde el punto de penalti',
        'params': dict(x=108, y=40, body_part='Right Foot',
                      technique='Normal', under_pressure=False,
                      shot_first_time=False, minute=30)
    },
    {
        'descripcion': 'Disparo lejano con pie izquierdo bajo presion',
        'params': dict(x=90, y=30, body_part='Left Foot',
                      technique='Normal', under_pressure=True,
                      shot_first_time=False, minute=75)
    },
    {
        'descripcion': 'Volea de pie derecho cerca del area pequeña',
        'params': dict(x=112, y=36, body_part='Right Foot',
                      technique='Volley', under_pressure=False,
                      shot_first_time=True, minute=45)
    },
    {
        'descripcion': 'Disparo lejano desde fuera del area',
        'params': dict(x=80, y=40, body_part='Right Foot',
                      technique='Normal', under_pressure=False,
                      shot_first_time=False, minute=20)
    },
]

for caso in casos:
    xg = predecir_xg(**caso['params'])
    distancia = calcular_distancia(caso['params']['x'], caso['params']['y'])
    angulo    = calcular_angulo(caso['params']['x'], caso['params']['y'])
    print(f"\n{caso['descripcion']}")
    print(f"  Posicion: x={caso['params']['x']}, y={caso['params']['y']}")
    print(f"  Distancia al gol: {distancia:.1f} | Angulo: {angulo:.1f} grados")
    print(f"  xG estimado: {xg:.4f} ({xg*100:.1f}% de probabilidad de gol)")

EJEMPLOS DE PREDICCION xG

Remate de cabeza al borde del area
  Posicion: x=105, y=40
  Distancia al gol: 15.0 | Angulo: 29.9 grados
  xG estimado: 0.0377 (3.8% de probabilidad de gol)

Disparo con pie derecho desde el punto de penalti
  Posicion: x=108, y=40
  Distancia al gol: 12.0 | Angulo: 36.9 grados
  xG estimado: 0.2758 (27.6% de probabilidad de gol)

Disparo lejano con pie izquierdo bajo presion
  Posicion: x=90, y=30
  Distancia al gol: 31.6 | Angulo: 13.7 grados
  xG estimado: 0.0180 (1.8% de probabilidad de gol)

Volea de pie derecho cerca del area pequeña
  Posicion: x=112, y=36
  Distancia al gol: 8.9 | Angulo: 45.0 grados
  xG estimado: 0.2635 (26.4% de probabilidad de gol)

Disparo lejano desde fuera del area
  Posicion: x=80, y=40
  Distancia al gol: 40.0 | Angulo: 11.4 grados
  xG estimado: 0.0090 (0.9% de probabilidad de gol)


Se prueba la función predecir_xg() con cinco tipos diferentes de disparos que suelen hacerse en un partido de fútbol: remate de cabeza al borde del área, penalti, disparo lejano bajo presión, volea cerca del área y disparo lejano fuera del área. Lo que se busca es verificar que el modelo predice de manera lógica y coherente con cómo funciona el fútbol. Los resultados lo confirman: el penalti tiene la mayor probabilidad de ser anotado (27.6%), mientras que los disparos desde lejos o bajo presión tienen una probabilidad mucho más baja (entre 0.9% y 1.8%.

In [5]:
print('PREDICCION INTERACTIVA DE xG')
print('=' * 40)
print('Introduce los datos del disparo:')
print('(Sistema de coordenadas StatsBomb: campo 120x80)')
print('(Porteria rival en x=120, centro en y=40)\n')

try:
    x     = float(input('Coordenada x del disparo (0-120): '))
    y     = float(input('Coordenada y del disparo (0-80):  '))
    bp    = input('Parte del cuerpo (Right Foot / Left Foot / Head / Other): ')
    tech  = input('Tecnica (Normal / Volley / Lob / Half Volley / Overhead Kick / Diving Header): ')
    pres  = input('Bajo presion? (s/n): ').strip().lower() == 's'
    first = input('Disparo a la primera? (s/n): ').strip().lower() == 's'
    mins  = int(input('Minuto del partido (0-90): '))

    xg = predecir_xg(x, y, bp, tech, pres, first, mins)
    dist = calcular_distancia(x, y)
    ang  = calcular_angulo(x, y)

    print(f'\n{"="*40}')
    print(f'RESULTADO:')
    print(f'  Distancia al gol: {dist:.1f}')
    print(f'  Angulo de disparo: {ang:.1f} grados')
    print(f'  xG estimado: {xg:.4f}')
    print(f'  Probabilidad de gol: {xg*100:.1f}%')
    print(f'{"="*40}')

except Exception as e:
    print(f'Error en la entrada: {e}')

PREDICCION INTERACTIVA DE xG
Introduce los datos del disparo:
(Sistema de coordenadas StatsBomb: campo 120x80)
(Porteria rival en x=120, centro en y=40)

Coordenada x del disparo (0-120): 100
Coordenada y del disparo (0-80):  60
Parte del cuerpo (Right Foot / Left Foot / Head / Other): Right foot
Tecnica (Normal / Volley / Lob / Half Volley / Overhead Kick / Diving Header): Volley
Bajo presion? (s/n): s
Disparo a la primera? (s/n): s
Minuto del partido (0-90): 70

RESULTADO:
  Distancia al gol: 28.3
  Angulo de disparo: 11.5 grados
  xG estimado: 0.0027
  Probabilidad de gol: 0.3%


Se crea una forma sencilla y fácil de usar para que cualquier persona pueda ingresar los datos de un disparo real (coordenadas, parte del cuerpo, técnica, presión, minuto). Con esta información, el sistema calcula de inmediato cuánta probabilidad hay de que ese disparo termine en gol.

In [6]:
import cloudpickle

with open(ruta + 'funcion_xg.pkl', 'wb') as f:
    cloudpickle.dump(predecir_xg, f)

print('Funcion predecir_xg() guardada como funcion_xg.pkl')
print('')
print('En un entorno de produccion, esta funcion se integraria')
print('en una API REST (FastAPI/Flask) que recibiria los atributos')
print('del disparo via HTTP y devolveria el xG en formato JSON.')
print('')
print('Ejemplo de respuesta JSON:')
print('  {')
print('    "x": 108, "y": 40,')
print('    "body_part": "Right Foot",')
print('    "technique": "Normal",')
print('    "xg_estimado": 0.2341,')
print('    "probabilidad_gol": "23.4%"')
print('  }')

Funcion predecir_xg() guardada como funcion_xg.pkl

En un entorno de produccion, esta funcion se integraria
en una API REST (FastAPI/Flask) que recibiria los atributos
del disparo via HTTP y devolveria el xG en formato JSON.

Ejemplo de respuesta JSON:
  {
    "x": 108, "y": 40,
    "body_part": "Right Foot",
    "technique": "Normal",
    "xg_estimado": 0.2341,
    "probabilidad_gol": "23.4%"
  }


Se guarda la función predecir_xg() en un archivo funcion_xg.pkl usando cloudpickle. También se añade, que en un entorno real de trabajo, esta función se integraría en un sistema que recibe solicitudes a través de una API (como FastAPI o Flask). Esta sería una forma de comunicarse con el programa por internet. Este sistema recibiría los datos, usaría la función para hacer predicciones y devolvería el resultado en un formato que puede entender cualquier programa (JSON). De modo que así se daría por finalizando el proceso productivización y se podría usar el modelo en la realidad.